# Complex CNN — kiểm tra nhanh quá trình train

Notebook này **không train lại 200 epoch**. Nó chỉ train **một batch** để kiểm tra các bước nạp dữ liệu, dự đoán, tính loss, cập nhật trọng số và lưu model đều chạy được.

Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** rồi chạy lần lượt các cell từ trên xuống.

In [ ]:
# Tải code và dataset từ GitHub.
%cd /content
!git clone --depth 1 https://github.com/oopsvananh/deep_learning-waste_classification.git
%cd /content/deep_learning-waste_classification

In [ ]:
# Colab đã có PyTorch; chỉ cài các thư viện còn lại.
%pip install -q pandas scikit-learn pillow

import torch

assert torch.cuda.is_available(), "Hãy bật T4 GPU trong Runtime settings."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Dùng model và cách xử lý dữ liệu có sẵn trong repository.
from models.model_2.train_complex_cnn import ComplexCNN
from preprocess.preprocessing import CLASSES, compute_class_weights, make_dataloaders

train_loader, _, _ = make_dataloaders()
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

model = ComplexCNN(num_classes=len(CLASSES)).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=compute_class_weights().to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# Một bước train: dự đoán → tính loss → cập nhật trọng số.
model.train()
optimizer.zero_grad()
outputs = model(images)
loss = loss_fn(outputs, labels)
loss.backward()
optimizer.step()

print(f"Smoke test thành công | loss = {loss.item():.4f}")

In [ ]:
# Lưu thử checkpoint; đây không phải model chính thức.
torch.save(model.state_dict(), "smoke_checkpoint.pth")
print("Đã lưu smoke_checkpoint.pth")
print("Model chính thức: weights/complex_cnn_best_epoch187.pth")